# pre/post backscatter + coherence (Sentinel-1 SLC)

Four SLC products — two before an event, two after — to two GeoTIFFs over an
area of interest: `<name>_pre.tif` and `<name>_post.tif`, each with
`gamma0_VH`, `gamma0_VV`, `coh_VH`, `coh_VV` on the same 10 m grid.

This notebook **drives** `pre_post_backscatter_coherence.py`, it does not
mirror it: the logic stays in the module, here are the parameters and the
calls. Everything below is also one command line —
`python rosarium.py slc --pre1 ... --output zta1` — and `README.md` explains
the steps, the image roles and where the outputs land.

**Before running**: SNAP must be installed (cell 2 says whether `gpt` was
found), and the four products downloaded, e.g. with
`python rosarium.py download`.

| Cell | What it does |
| --- | --- |
| 1 | Imports the module (run the notebook from its own folder). |
| 2 | Prints where SNAP's `gpt` was found and the default memory settings. |
| 3 | Parameters: the four products, the AOI, the run name, the gpt settings. |
| 4 | Which sub-swaths and bursts the AOI needs — a few seconds, no processing. |
| 5 | Runs the whole pipeline (long: 10-30 min) and prints the two products. |
| 6 | Opens the result: bands, size, and a quick look at the pre gamma0_VH. |

In [ ]:
# === 1. The module sitting next to this notebook ===
from pre_post_backscatter_coherence import main_preprocess

from features.snap_gpt.snap_gpt import (
    DEFAULT_GPT, DEFAULT_GPT_OPTIONS, GptOptions, polygon_to_swaths_bursts,
)

In [ ]:
# === 2. Is SNAP there? ===
# Looked for in SNAP_GPT, then on the PATH, then in the usual install folders.
# If not found, install SNAP and pass GPT_PATH below (cell 3).
print("gpt:", DEFAULT_GPT or "NOT FOUND - install SNAP, or set GPT_PATH in cell 3")
print("default settings:", DEFAULT_GPT_OPTIONS)

In [ ]:
# === 3. Parameters ===

# The four SLC products (.SAFE folder or .zip), same relative orbit, in order.
# pre2 is the backscatter master, post1 its slave; pre1xpre2 and post1xpost2
# are the two coherence pairs.
PRE1  = r"C:\Users\guigu\Documents\pro_asus\rosarium\data\raw\vrac\S1A_..._pre1.SAFE"
PRE2  = r"C:\Users\guigu\Documents\pro_asus\rosarium\data\raw\vrac\S1A_..._pre2.SAFE"
POST1 = r"C:\Users\guigu\Documents\pro_asus\rosarium\data\raw\vrac\S1A_..._post1.SAFE"
POST2 = r"C:\Users\guigu\Documents\pro_asus\rosarium\data\raw\vrac\S1A_..._post2.SAFE"

# Area of interest, lon/lat (EPSG:4326): an inline WKT string, or a path to a
# WKT / GeoJSON file — the <name>_aoi.geojson the webmap writes works as is.
AOI = r"C:\Users\guigu\Documents\pro_asus\rosarium\data\utils\list_aoi.geojson"

# Run name: the products land in data/preprocessed/pre_post/<OUTPUT_NAME>/.
# A path instead of a name writes in that folder, using its last segment as
# prefix. That folder is shared with the GRD pipeline: name the run for what
# it holds.
OUTPUT_NAME = "zta1_slc"

# SNAP's gpt: None lets the module find it (cell 2), or give a full path.
GPT_PATH = DEFAULT_GPT

# Heap / cache / threads / tile size handed to every gpt call. The defaults
# suit a 32 GB / 8-core machine; on a 16 GB laptop try
# GptOptions(xmx="10G", cache="3G", threads=4). See the snap_gpt README.
GPT_OPTIONS = DEFAULT_GPT_OPTIONS

In [ ]:
# === 4. What the AOI needs: sub-swaths and burst range (seconds, no processing) ===
# Coarse mode, as the pipeline runs it. Several sub-swaths here means the
# graphs run once per sub-swath and the tiles are mosaicked at the end.
swaths = polygon_to_swaths_bursts(PRE2, AOI)
for s in swaths:
    print(f"{s['subswath']}: bursts {s['first_burst']}-{s['last_burst']}")
print(f"{len(swaths)} sub-swath(s)" + (" -> mosaic at the end" if len(swaths) > 1 else ""))

In [ ]:
# === 5. The pipeline: backscatter, coherence pre, coherence post, gathering, mosaic ===
# Long (10-30 min on a few bursts). SNAP's log goes to the terminal running
# the kernel, not here — for a live log, run the command line instead:
#   python rosarium.py slc --pre1 ... --pre2 ... --post1 ... --post2 ... --aoi ... --output ...
products = main_preprocess(
    pre1=PRE1, pre2=PRE2, post1=POST1, post2=POST2,
    aoi=AOI, output_name=OUTPUT_NAME,
    gpt_path=GPT_PATH, gpt_options=GPT_OPTIONS,
)
products

In [ ]:
# === 6. A look at what came out ===
from osgeo import gdal

gdal.UseExceptions()
for role, path in products.items():
    ds = gdal.Open(path)
    bands = [ds.GetRasterBand(i + 1).GetDescription() for i in range(ds.RasterCount)]
    print(f"{role:4}  {ds.RasterXSize} x {ds.RasterYSize} px  {bands}")
    ds = None

# Quick look at the pre gamma0_VH, in dB (nodata 0 -> NaN so it does not
# flatten the colour scale)
import matplotlib.pyplot as plt
import numpy as np

ds = gdal.Open(products["pre"])
arr = ds.GetRasterBand(1).ReadAsArray().astype("float32")
ds = None
arr[arr == 0] = np.nan
plt.figure(figsize=(8, 8))
plt.imshow(10 * np.log10(arr), cmap="gray")
plt.title("pre - gamma0_VH (dB)")
plt.colorbar(shrink=0.7)
plt.axis("off")
plt.show()